TASK C (Epidemic Models)

This notebook investigates how suspicious or controversial behaviour might propagate through Wikidata editor networks.

The task is approached as a network-based propagation problem. The technical assumption used throughout is that if one or more editors show unusually high activity in the same discussion context, then neighbouring or structurally similar editors may have an elevated chance of also being involved.

To examine this, three complementary approaches are implemented:

1. Distance-based approach - Uses shortest-path distance from suspicious editors to estimate how far possible propagation may have reached.

2. Neighbourhood-based approach - Uses immediate neighbours and local overlap to identify editors who are structurally close to suspicious editors.

3. Epidemic-style propagation approach - Uses a simple network diffusion model to simulate how suspicious behaviour may spread through the graph.

These methods are then compared in order to build a priority list of editors to check first.

In [2]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from itertools import combinations
import random

Step 1: Reconstruct the Networks

The same graph construction logic from Tasks A and B is reused so that propagation analysis is performed on the same network definition.

Nodes represent editors, and undirected edges represent co-participation in the same page-thread discussion context.

In [3]:
datasets = {
    "small": "../data/REQUEST_A_QUERY.csv",
    "medium": "../data/USERS.csv",
    "large": "../data/REQUEST_FOR_DELETION.csv"
}

def build_graph(df, user_col="username", context_cols=("page_name", "thread_subject"), max_group_size=100):
    df = df.dropna().copy()
    df[user_col] = df[user_col].astype(str).str.strip()
    for col in context_cols:
        df[col] = df[col].astype(str).str.strip()

    G = nx.Graph()

    for context, group in df.groupby(list(context_cols)):
        users = group[user_col].dropna().unique()

        if len(users) > max_group_size:
            users = users[:max_group_size]

        for u1, u2 in combinations(users, 2):
            G.add_edge(u1, u2)

    return G

graphs = {}
for name, path in datasets.items():
    df = pd.read_csv(path)
    graphs[name] = build_graph(df)

print("Networks reconstructed successfully.")

Networks reconstructed successfully.


Step 2: Define the Working Hypothesis

The technical hypothesis used in this notebook is:

"If one or more editors display suspiciously high activity, then editors that are closer to them in the network, share neighbours with them, or are repeatedly reached by diffusion processes should be treated as higher-priority candidates for further checking."

This is not treated as a proven fact. Instead, it is used as a modelling assumption that can be explored using different network-based methods.

Step 3: Select Seed Editors

The coursework scenario refers to one or two initially suspicious editors. To make the analysis reproducible, seed editors are selected from the largest connected component of each network.

For demonstration, the first analysis will use two seed editors. Later sections can compare the one-seed and two-seed cases.

In [4]:
seed_nodes = {}

for name, G in graphs.items():
    lcc_nodes = list(max(nx.connected_components(G), key=len))
    random.seed(42)
    chosen = random.sample(lcc_nodes, 2)
    seed_nodes[name] = chosen

seed_nodes

{'small': ['TomT0m', 'Jheald'],
 'medium': ['DSGalaktos', 'Chicbyaccident'],
 'large': ['JAn Dudík', 'Ortheus099']}

The selected seed editors serve as the starting point for all three methods. They are not assumed to be confirmed trolls; they are treated as initial suspicious cases.

Step 4:

Method 1: Distance-Based Plausibility Assessment

The first method uses shortest-path distance from the suspicious editors.

Technical idea:
- If suspicious behaviour has not propagated far, then only editors very close to the seed editors should be considered plausible next cases.
- If many editors are already at short path distance from the seeds, then non-propagation becomes less plausible.

This method therefore treats network distance as a proxy for potential reach.

In [5]:
distance_results = []

for name, G in graphs.items():
    seeds = seed_nodes[name]

    min_distances = {}
    for node in G.nodes():
        dists = []
        for s in seeds:
            try:
                dists.append(nx.shortest_path_length(G, source=s, target=node))
            except nx.NetworkXNoPath:
                pass
        if dists:
            min_distances[node] = min(dists)

    ranked = sorted(min_distances.items(), key=lambda x: x[1])
    top_distance = ranked[:15]

    for node, dist in top_distance:
        distance_results.append([name, node, dist])

distance_df = pd.DataFrame(distance_results, columns=["Dataset", "Node", "Min Distance From Seeds"])
distance_df.head(20)

,Dataset,Node,Min Distance From Seeds
0,small,TomT0m,0
1,small,Jheald,0
2,small,Jura1,1
3,small,Tagishsimon,1
4,small,Larske,1
5,small,Airon90,1
6,small,Dipsacus fullonum,1
7,small,Matěj Suchánek,1
8,small,Edgars2007,1
9,small,Spinster,1


The table ranks editors by minimum shortest-path distance from the suspicious seeds. Smaller values indicate stronger structural proximity and therefore higher immediate plausibility of involvement under the distance-based hypothesis.

Method 2: Neighbourhood-Based Prioritisation

The second method uses local structure rather than global distance.

Technical idea:
- Editors directly adjacent to suspicious seeds may have elevated relevance because they share a discussion neighbourhood.
- Editors with greater overlap in neighbours may also be structurally similar to the seeds.

This method therefore prioritises nodes using:
1. adjacency to seed editors
2. number of shared neighbours with the seeds

In [6]:
neighbour_results = []

for name, G in graphs.items():
    seeds = seed_nodes[name]
    seed_set = set(seeds)

    scores = {}

    for node in G.nodes():
        if node in seed_set:
            continue

        node_neigh = set(G.neighbors(node))
        shared = 0
        direct_links = 0

        for s in seeds:
            seed_neigh = set(G.neighbors(s))
            shared += len(node_neigh & seed_neigh)
            if G.has_edge(node, s):
                direct_links += 1

        scores[node] = (direct_links, shared)

    ranked = sorted(scores.items(), key=lambda x: (x[1][0], x[1][1]), reverse=True)[:15]

    for node, (direct_links, shared) in ranked:
        neighbour_results.append([name, node, direct_links, shared])

neighbour_df = pd.DataFrame(
    neighbour_results,
    columns=["Dataset", "Node", "Direct Links To Seeds", "Shared Neighbours"]
)

neighbour_df.head(20)

,Dataset,Node,Direct Links To Seeds,Shared Neighbours
0,small,Tagishsimon,2,79
1,small,Jura1,2,64
2,small,MisterSynergy,2,61
3,small,Matěj Suchánek,2,60
4,small,Dipsacus fullonum,2,51
5,small,Larske,2,43
6,small,Pigsonthewing,2,35
7,small,Bouzinac,2,33
8,small,Andrew Gray,2,32
9,small,Luitzen,2,29


This ranking prioritises editors who are directly linked to the suspicious seeds and who also share similar local neighbourhoods. It is a local-structure method rather than a diffusion model.

Method 3: Epidemic-Style Propagation Simulation

The third method treats suspicious behaviour as something that can spread probabilistically through network edges.

A simplified SI-style process is used:
- seed editors start in the suspicious state
- at each step, a suspicious editor can activate a neighbour with probability `p_infect`
- once activated, a node remains suspicious for the remainder of the run

This is not intended as a literal behavioural truth. It is a technical simulation used to estimate which editors are repeatedly reached under a propagation-based assumption.

In [7]:
def run_si_simulation(G, seeds, p_infect=0.2, steps=5, rng_seed=42):
    rng = random.Random(rng_seed)
    infected = set(seeds)
    newly_infected = set(seeds)

    for _ in range(steps):
        next_wave = set()
        for node in newly_infected:
            for nbr in G.neighbors(node):
                if nbr not in infected and rng.random() < p_infect:
                    next_wave.add(nbr)

        if not next_wave:
            break

        infected.update(next_wave)
        newly_infected = next_wave

    return infected

Repeating the Simulation as a single simulation run can vary because propagation is probabilistic. To obtain a more stable prioritisation, the simulation is repeated multiple times and the frequency with which each editor becomes infected is recorded.

Editors who are infected more often across runs are treated as higher-priority candidates under the epidemic-style hypothesis.

In [8]:
simulation_results = []

num_runs = 30

for name, G in graphs.items():
    seeds = seed_nodes[name]
    infection_counts = {node: 0 for node in G.nodes()}

    for run in range(num_runs):
        infected = run_si_simulation(G, seeds, p_infect=0.2, steps=5, rng_seed=run)
        for node in infected:
            infection_counts[node] += 1

    ranked = sorted(
        [(node, count / num_runs) for node, count in infection_counts.items() if node not in seeds],
        key=lambda x: x[1],
        reverse=True
    )[:15]

    for node, prob in ranked:
        simulation_results.append([name, node, prob])

simulation_df = pd.DataFrame(
    simulation_results,
    columns=["Dataset", "Node", "Estimated Infection Probability"]
)

simulation_df.head(20)

,Dataset,Node,Estimated Infection Probability
0,small,Jura1,1.000000
1,small,Tagishsimon,1.000000
2,small,Larske,1.000000
3,small,Dipsacus fullonum,1.000000
4,small,Matěj Suchánek,1.000000
5,small,Edgars2007,1.000000
6,small,Jklamo,1.000000
7,small,Pasleim,1.000000
8,small,Daniel Mietchen,1.000000
9,small,VIGNERON,1.000000


The estimated infection probability is the proportion of simulation runs in which an editor was reached by the propagation process. Higher values indicate higher priority under the epidemic-style model.

Step 5: Comparing the Three Methods

The three methods do not measure exactly the same thing:
- distance-based ranking measures structural closeness
- neighbourhood-based ranking measures local similarity and direct exposure
- simulation-based ranking measures repeated reachability under diffusion

Comparing them helps identify editors who are consistently prioritised under multiple assumptions.

In [9]:
comparison_tables = {}

for name in graphs.keys():
    d_sub = distance_df[distance_df["Dataset"] == name].copy()
    n_sub = neighbour_df[neighbour_df["Dataset"] == name].copy()
    s_sub = simulation_df[simulation_df["Dataset"] == name].copy()

    d_sub["Distance Rank"] = range(1, len(d_sub) + 1)
    n_sub["Neighbour Rank"] = range(1, len(n_sub) + 1)
    s_sub["Simulation Rank"] = range(1, len(s_sub) + 1)

    merged = d_sub[["Node", "Distance Rank"]].merge(
        n_sub[["Node", "Neighbour Rank"]], on="Node", how="outer"
    ).merge(
        s_sub[["Node", "Simulation Rank"]], on="Node", how="outer"
    )

    comparison_tables[name] = merged.sort_values(
        by=["Distance Rank", "Neighbour Rank", "Simulation Rank"],
        na_position="last"
    )

comparison_tables["small"].head(20)

,Node,Distance Rank,Neighbour Rank,Simulation Rank
22,TomT0m,1.0,NaN,NaN
8,Jheald,2.0,NaN,NaN
10,Jura1,3.0,2.0,1.0
21,Tagishsimon,4.0,1.0,2.0
11,Larske,5.0,6.0,3.0
0,Airon90,6.0,15.0,NaN
4,Dipsacus fullonum,7.0,5.0,4.0
13,Matěj Suchánek,8.0,4.0,5.0
5,Edgars2007,9.0,NaN,6.0
19,Spinster,10.0,NaN,NaN


Editors that appear near the top across multiple rankings can be treated as especially plausible candidates for follow-up checking.

Step 6: One-Seed vs Two-Seed Scenario

The coursework notes that prioritisation may differ depending on whether one suspicious editor is observed or two. This is tested by repeating the simulation using one seed instead of two.

This comparison is important because two suspicious editors may indicate either:
- a stronger local signal in one community, or
- multiple entry points for propagation

In [10]:
one_vs_two_results = []

for name, G in graphs.items():
    seeds_two = seed_nodes[name]
    seed_one = [seeds_two[0]]

    infected_one = run_si_simulation(G, seed_one, p_infect=0.2, steps=5, rng_seed=42)
    infected_two = run_si_simulation(G, seeds_two, p_infect=0.2, steps=5, rng_seed=42)

    one_vs_two_results.append([
        name,
        len(seed_one),
        len(infected_one),
        len(seeds_two),
        len(infected_two)
    ])

one_vs_two_df = pd.DataFrame(
    one_vs_two_results,
    columns=["Dataset", "One Seed Count", "Reached From One Seed", "Two Seed Count", "Reached From Two Seeds"]
)

one_vs_two_df

,Dataset,One Seed Count,Reached From One Seed,Two Seed Count,Reached From Two Seeds
0,small,1,254,2,257
1,medium,1,2725,2,2747
2,large,1,3943,2,3921


This comparison provides a simple indication of how much the suspected spread expands when the process starts from two suspicious editors rather than one.

 Step 7: Technical Summary

The notebook implemented three complementary approaches for identifying editors who should be prioritised after suspicious activity is observed:

1. shortest-path proximity
2. local neighbourhood similarity
3. epidemic-style probabilistic diffusion

These methods provide different technical views of possible propagation. The report will interpret these results in relation to cascading behaviour, epidemic spread, and prioritisation strategies discussed in the module.